# Fixed-chunk speech translation: from audio to measured results

How does English speech become German text, and how long does the listener wait?
**Automatic speech recognition (ASR)** first turns English audio into English text.
**Machine translation (MT)** then translates that text into German. These stages run
sequentially, forming a **cascade**: English audio → Whisper ASR → English text → Marian MT → German text.

Start with the saved English/German examples and their timing below. Then compare the
output with supplied reference text and read the results. Source listings and the
evaluation repair are optional reading in the appendix.

This notebook was prepared after measurement. **Run All reads local files only:** no
model loading, audio replay, downloads or paid API requests. See `CODE_GUIDE.md` for
the implementation map.

In [1]:
from pathlib import Path
import ast
import json
import sys
from IPython.display import Code, JSON, Markdown, display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "CODE_GUIDE.md").is_file() and (p / "quality-control").is_dir())
sys.path.insert(0, str(ROOT / "quality-control"))
import qc_notebook as qc

def read(path):
    return json.loads(path.read_text(encoding="utf-8"))

def read_lines(path):
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]

def show_source(relative_path, name):
    text = (ROOT / relative_path).read_text(encoding="utf-8")
    node = next(n for n in ast.walk(ast.parse(text))
                if isinstance(n, (ast.FunctionDef, ast.ClassDef)) and n.name == name)
    display(Code("\n".join(text.splitlines()[node.lineno-1:node.end_lineno]), language="python"))

print("Loaded local submission:", ROOT.name)


Loaded local submission: TinyML-code-submission


In [2]:
MAIN = ROOT / "TinyML-main-experiment-2026-10-18"
EVIDENCE = ROOT / "evidence/main"
EVALUATION = EVIDENCE / "evaluation-main-fixed"
scores = read(ROOT / "analysis/main/data/main-scores.json")
summary = read(ROOT / "analysis/main/data/main-summary.json")

## 1. Follow one English/German example

This example shows the first 12 seconds of one recording, processed on the Raspberry Pi using Whisper `tiny.en`. The recording is divided into **chunks**: consecutive blocks of four seconds of audio.

The table below contains one row per chunk. “English ASR” shows the English text recognized by Whisper. This text, including any recognition errors, is then passed to the Marian translation model. “German MT” shows the resulting German translation.

The chunk boundaries occur every four seconds, regardless of where sentences begin or end. A sentence can therefore be split across several rows.

Keep three different units in mind:

| Unit | Purpose |
|---|---|
| Chunk | A 4- or 8-second processing block; the last block may be shorter. |
| Reference segment | A supplied comparison unit used for scoring; its boundaries do not control processing. |
| Quality control (QC) window | A 24-second passage inspected for translation fidelity, with neighboring context. |

The following cell loads saved results from the first pass of the Raspberry Pi / `tiny.en` / 4-second configuration. It displays existing output; it does not run either model.

In [3]:
run = next(r for r in scores["runs"] if r["device"] == "pi" and r["asr"] == "tiny.en"
           and r["chunk_s"] == 4 and r["repetition"] == 1)
job = EVIDENCE / "runs" / run["device"] / "jobs" / run["id"]
marker = read(job / "complete.json")
attempt = job / marker["attempt"]
chunks = read_lines(attempt / "chunks.jsonl")
print(run["key"])
qc.table({"Chunk": c["chunk_index"], "Audio interval (s)": f'{c["source_start_s"]:.1f}–{c["source_end_s"]:.1f}',
          "English ASR": c["english"], "German MT": c["german"]} for c in chunks[:3])


pi/rxrToXvRyM-tiny-4s-r1


Chunk,Audio interval (s),English ASR,German MT
0,0.0–4.0,"Hi, I'm Mara, and today I'll be talking about our paper marked percentages.","Hallo, ich bin Mara, und heute rede ich von unserer Zeitung, die deutliche Prozentsätze hat."
1,4.0–8.0,using natural language prompts to measure stereotypes in language models.,Verwendung natürlicher Sprachanweisungen zur Messung von Stereotypen in Sprachmodellen.
2,8.0–12.0,This work is done in collaboration with SN Durham-Osh and Dandruff-E.,Diese Arbeit wird in Zusammenarbeit mit SN Durham-Osh und Dandruff-E durchgeführt.


Each row shows a separate recognition-and-translation step. Whisper receives only that row’s audio chunk, and Marian receives only the English text recognized from that chunk. The system does not wait for the next chunk to complete an unfinished sentence, and it does not revise an earlier translation when more audio arrives.

This also explains how recognition errors can propagate: Marian may faithfully translate the English text it receives even when that text differs from what the speaker actually said.

## 2. When does the German output become available?

The experiment replays a saved recording at its original speed, so audio becomes available as if someone were speaking live. Processing a chunk begins only when all its audio has arrived and the previous chunk has finished processing.

Whisper first recognizes the English speech, then Marian translates the recognized text. Once both stages finish, the complete German translation for that chunk becomes available together.

The table measures time in seconds from the start of playback. “Audio ready” marks the end of the chunk’s audio. “Start delay” is the additional time before processing begins. It can include waiting for the previous chunk and small scheduling delays. In these three examples it is less than 0.001 seconds, so processing starts almost immediately after the audio is ready. Times are displayed to three decimal places; a displayed start delay of 0.000 seconds means less than half a millisecond after rounding, rather than exactly zero.

In [4]:
qc.table({"Chunk": c["chunk_index"], "Audio ready (s)": f'{c["source_end_s"]:.3f}',
          "Processing start (s)": f'{c["processing_start_s"]:.3f}',
          "Output available (s)": f'{c["emitted_s"]:.3f}',
          "Start delay (s)": f'{c["queue_wait_s"]:.3f}',
          "Processing (s)": f'{c["process_s"]:.3f}'} for c in chunks[:3])
for c in chunks:
    assert abs(c["process_s"] - (c["emitted_s"] - c["processing_start_s"])) < 1e-5
    assert c["processing_start_s"] + 1e-5 >= c["source_end_s"]
print("Saved chunk timing identities checked.")

Chunk,Audio ready (s),Processing start (s),Output available (s),Start delay (s),Processing (s)
0,4.000,4.000,5.584,0.000,1.584
1,8.000,8.000,9.309,0.000,1.309
2,12.000,12.000,13.564,0.000,1.564


Saved chunk timing identities checked.


The first chunk contains audio from 0 to 4 seconds. Processing starts almost immediately after the four-second mark and takes about 1.584 seconds. Its German translation therefore becomes available approximately 5.584 seconds after playback begins.

For latency evaluation, we assign output times using two **timing conventions**:

- **Computation-unaware (CU):** assign the time when the chunk’s audio is complete—4.000 seconds in this example. This represents an idealized system with instantaneous processing.
- **Computation-aware (CA):** assign the measured time when the German translation actually becomes available—5.584 seconds in this example. This includes processing time and any delay before processing starts.

Both include waiting for the chunk’s audio to arrive. Their difference is whether processing and subsequent delays are included.

The evaluator needs a timestamp for every German output word. Because this system makes the whole chunk’s translation available together, every word in that chunk receives the same timestamp. The resulting lists contain 4.000 for every word under CU and 5.584 for every word under CA.

These timestamps tell the evaluator **when words become available**. They are inputs to the LongYAAL calculation introduced later; they are not themselves the final translation-delay scores.

In [5]:
first = chunks[0]
qc.table({"German output word": word,
          "CU timestamp (s)": f'{first["source_end_s"]:.3f}',
          "CA timestamp (s)": f'{first["emitted_s"]:.3f}'}
         for word in first["words"])

German output word,CU timestamp (s),CA timestamp (s)
"Hallo,",4.000,5.584
ich,4.000,5.584
bin,4.000,5.584
"Mara,",4.000,5.584
und,4.000,5.584
heute,4.000,5.584
rede,4.000,5.584
ich,4.000,5.584
von,4.000,5.584
unserer,4.000,5.584


Every word in the table has the same CU timestamp and the same CA timestamp because the whole chunk’s translation becomes available at once. Individual words are not made available as they are generated.

## 3. Model settings and processing cost

In the main experiment, Whisper is configured to transcribe English speech into English text. Marian then translates that text into German.

Each audio chunk is processed independently: recognized text from earlier chunks is not supplied as context for later chunks. This simplifies processing, but can make sentences split across chunk boundaries harder to recognize and translate.

The recorded model settings and processing times are shown below.

- **Beam** is the number of candidate text sequences retained during decoding. ASR beam 1 keeps one; MT beam 4 explores four. The experiment wrapper supplies MT beam 4, overriding the older low-level runner's default of 1.
- **Tokenization** converts text into model units, which may be words or parts of words. MT timing includes tokenization, translation and decoding back to readable text.
- **`int8_float32`** uses 8-bit integer quantization with 32-bit floating-point operations for the unquantized parts.
- **Warm-up** consists of preliminary model calls to prepare execution before measurement. Model loading and warm-up are outside the replay processing measure.

**Real-time factor (RTF)** is processing time divided by audio duration. For example,
2 seconds of processing for 10 seconds of audio gives **0.2**. It excludes deliberate
waiting for incoming audio. Lower means less processing per second of audio; it does
not directly tell us how long a listener waits for a translation.

In [6]:
metadata = read(attempt / "metadata.json")
display(JSON({"recorded_decoding": metadata["decoding"],
              "effective_compute_type": metadata["effective_compute_type"]}, expanded=True))
duration = run["audio_duration_s"]
qc.table([{"Measure": name, "Seconds": round(run[field], 3), "RTF": round(run[field] / duration, 4)}
          for name, field in [("ASR", "total_asr_s"), ("MT", "total_mt_s"),
                              ("Total processing", "total_processing_s")]])


<IPython.core.display.JSON object>

Measure,Seconds,RTF
ASR,93.595,0.2198
MT,46.28,0.1087
Total processing,139.882,0.3285


ASR accounts for more processing time than MT in this saved run. Total processing
RTF is below 1, meaning less processing time than audio duration overall; individual
chunks can still take long enough to create a queue.

## 4. Compare predictions with references

A **reference** is the supplied comparison text. A **prediction**, also called a
**hypothesis**, is the system's output. The evaluator groups the continuous German
output into units corresponding to the reference segments (**resegmentation**), while
keeping the output-word timestamps. These scoring units differ from processing chunks.

In [7]:
segments = read(EVALUATION / run["segments_file"])
qc.table({"English reference": s["src"], "German reference": s["reference"],
          "German prediction": s["prediction"]} for s in segments[:2])

English reference,German reference,German prediction
"""Hi, I'm Myra and today I'll be talking about our paper """"Marked Personas: Using Natural Language Prompts to Measure Stereotypes in Language Models.""""","Hallo, mein Name ist Myra und ich werde heute über unsere Arbeit „Marked Personas: Using Natural Language Prompts to Measure Stereotypes in Language Models“ sprechen.","Hallo, ich bin Mara, und heute rede ich von unserer Zeitung, die deutliche Prozentsätze hat. Verwendung natürlicher Sprachanweisungen zur Messung von Stereotypen in Sprachmodellen."
This work is done in collaboration with Esin Durmus and Dan Jurafsky.,Es handelt sich dabei eine Zusammenarbeit mit Esin Durmus und Dan Jurafsky.,Diese Arbeit wird in Zusammenarbeit mit SN Durham-Osh und Dandruff-E durchgeführt.


Compare the German reference and prediction within each row. The prediction reflects
both recognition and translation, whereas the reference is supplied independently.
These rows are reference segments, so their boundaries need not match the earlier chunk table.

The quality metrics answer different questions:

| Metric | What it measures | Direction |
|---|---|---|
| WER (word error rate) | English recognition errors: (substitutions + deletions + insertions) / reference-word count. | Lower is better. |
| chrF2 | Character-sequence overlap with the German reference, with recall weighted more heavily than precision. | Higher is better; it is not a percentage of correctly translated meaning. |
| COMET | A learned translation-quality metric using source text, translation and reference. Here the source is the official English reference, not ASR text. | Higher is better; it is not a percentage. |

For a small WER example, **one substitution, one deletion and one insertion against
ten reference words gives (1 + 1 + 1) / 10 = 30% WER**. A word deletion is a text-alignment
error; it is not automatically a meaningful translation omission. Meaning can be
expressed differently or recovered in context, which requires separate inspection.

**LongYAAL** measures translation delay using output timing and reference-based
evaluation. Lower means less measured delay. Its scores are in seconds, unlike the
processing RTF ratio. CU and CA are two timing conventions used with this metric, and both include
waiting for incoming audio. End-of-recording cutoffs can leave different eligible
segments under each timing convention; CA minus CU is therefore not pure computation time, and CA is not CU plus RTF.

## 5. The experiment behind the comparisons

Six full talks are crossed with tiny.en/base.en, 4-/8-second chunks and two devices,
with two passes each: 6 × 2 × 2 × 2 × 2 = 96 jobs. The independent content unit is a
recording. Two passes repeat execution, not content. Reference sentence boundaries
are reserved for evaluation.

In [8]:
qc.table({"Talk": t["id"], "Audio seconds": round(t["duration_s"], 3),
          "Reference segments": t["reference_segments"]} for t in scores["manifest"]["talks"])
print("Unique audio minutes:", round(sum(t["duration_s"] for t in scores["manifest"]["talks"]) / 60, 2))
print("Completed jobs:", scores["observed_runs"], "— expected:", scores["expected_runs"])
print("Configurations:", sorted({(r["asr"], r["chunk_s"]) for r in scores["runs"]}))

Talk,Audio seconds,Reference segments
rxrToXvRyM,425.808,52
EqmWoxNDIr,301.0,46
UOlPKyCVgg,299.179,32
krJSAnVcGR,304.127,43
ERmKpJPPDc,340.32,48
xhjBHGVOyQ,347.603,45


Unique audio minutes: 33.63
Completed jobs: 96 — expected: 96
Configurations: [('base.en', 4), ('base.en', 8), ('tiny.en', 4), ('tiny.en', 8)]


The talks differ in duration and number of reference segments. Repeating their
execution gives timing observations, while the unique audio remains 33.63 minutes
across six talks; 96 jobs do not represent 96 independent recordings.

## 6. Aggregate results and the quality–delay comparison

Within each pass, WER pools error and reference-word counts; RTF pools processing
seconds and audio duration. chrF2 is corpus-level, COMET averages segments, and
LongYAAL averages eligible segments. The following cells recompute WER, RTF and
LongYAAL aggregates from saved values, without rerunning the models or scorers.

In [9]:
selected = [r for r in scores["runs"] if r["device"] == "pi" and r["asr"] == "tiny.en"
            and r["chunk_s"] == 4 and r["repetition"] == 1]
saved_pass = next(p for p in summary["passes"] if p["device"] == "pi" and p["asr"] == "tiny.en"
                  and p["chunk_s"] == 4 and p["repetition"] == 1)
pooled_wer = 100 * sum(r["wer_errors"] for r in selected) / sum(r["wer_reference_words"] for r in selected)
pooled_rtf = sum(r["total_processing_s"] for r in selected) / sum(r["audio_duration_s"] for r in selected)
assert abs(pooled_wer - saved_pass["wer_pct"]) < 1e-9
assert abs(pooled_rtf - saved_pass["rtf_processing"]) < 1e-9
qc.table([{"Aggregate": "WER (%)", "Recomputed": pooled_wer, "Saved": saved_pass["wer_pct"]},
          {"Aggregate": "Processing RTF", "Recomputed": pooled_rtf, "Saved": saved_pass["rtf_processing"]}])
display(Markdown("The recomputed and saved values match. WER weights by reference-word count; RTF weights by audio duration, so neither is an unweighted average of talk-level rates."))
coverage = []
for timing in ("cu", "ca"):
    values = [v for r in selected for v in r["latency"][timing]["segment_scores_s"] if v is not None]
    mean = sum(values) / len(values)
    assert abs(mean - saved_pass[f"longyaal_{timing}_s"]) < 1e-9
    coverage.append({"Timing convention": timing.upper(), "Eligible segments": len(values), "LongYAAL (s)": mean})
qc.table(coverage)

Aggregate,Recomputed,Saved
WER (%),36.02219376867264,36.02219376867264
Processing RTF,0.3226230057028577,0.3226230057028577


The recomputed and saved values match. WER weights by reference-word count; RTF weights by audio duration, so neither is an unweighted average of talk-level rates.

Timing convention,Eligible segments,LongYAAL (s)
CU,264,2.15016208891201
CA,260,3.533417391310707


Compare both the delay and the eligible-segment count for CU and CA. CA includes
actual processing and queueing, but the two means can cover different segment sets.
The timestamp 5.584 seconds in the earlier example is an output time, not one of
these aggregate delay scores.

In [10]:
qc.table({"Device": c["device"], "ASR": c["asr"], "Chunk (s)": c["chunk_s"],
          "WER (%)": round(c["metrics"]["wer_pct"]["mean"], 2),
          "chrF2": round(c["metrics"]["chrf2"]["mean"], 2),
          "COMET": round(c["metrics"]["comet"]["mean"], 3),
          "Processing RTF": round(c["metrics"]["rtf_processing"]["mean"], 3),
          "CA LongYAAL (s)": round(c["metrics"]["longyaal_ca_s"]["mean"], 2)}
         for c in summary["configurations"])

Device,ASR,Chunk (s),WER (%),chrF2,COMET,Processing RTF,CA LongYAAL (s)
pi,tiny.en,4,36.02,46.68,0.559,0.322,3.53
pi,tiny.en,8,25.61,50.74,0.626,0.228,6.04
pi,base.en,4,26.36,50.26,0.602,0.516,4.2
pi,base.en,8,22.92,52.28,0.659,0.35,7.12
laptop,tiny.en,4,40.91,45.53,0.556,0.097,2.59
laptop,tiny.en,8,26.18,50.52,0.629,0.059,4.67
laptop,base.en,4,26.95,49.92,0.603,0.173,2.82
laptop,base.en,8,29.39,51.31,0.65,0.1,5.02


For each device and ASR model, moving from 4- to 8-second chunks raises chrF2 and
COMET but also raises CA LongYAAL: the quality improvement comes with more measured
delay. WER does not improve in every case (laptop/base.en is the exception). The
laptop has lower processing RTF than the Pi for each matching model/chunk setting.
Values are means over two passes, not estimates from additional independent talks.

## 7. The Whistle supplement

The separate system uses Whistle through its native C API, then the same Marian model. Native ASR thread/beam controls are compiled defaults. It retains independent outer chunks; it does not evaluate Whistle's built-in streaming policy. The recorded first laptop 8-second job began without AC power and remains in the results.


In [11]:
whistle = read(ROOT / "evidence/whistle/whistle-summary.json")
qc.table({"Device": c["device"], "Chunk (s)": c["chunk_s"],
          "WER (%)": round(c["metrics"]["wer_pct"]["mean"], 2),
          "chrF2": round(c["metrics"]["chrf2"]["mean"], 2),
          "COMET": round(c["metrics"]["comet"]["mean"], 3),
          "ASR RTF": round(c["metrics"]["asr_rtf"]["mean"], 3),
          "MT RTF": round(c["metrics"]["mt_rtf"]["mean"], 3),
          "CA LongYAAL (s)": round(c["metrics"]["longyaal_ca_s"]["mean"], 2)}
         for c in whistle["configurations"])

Device,Chunk (s),WER (%),chrF2,COMET,ASR RTF,MT RTF,CA LongYAAL (s)
pi,4,33.42,46.87,0.558,0.021,0.398,3.91
pi,8,26.85,50.13,0.607,0.02,0.381,7.44
laptop,4,33.48,46.9,0.558,0.037,0.019,2.4
laptop,8,26.76,50.04,0.606,0.134,0.019,5.41


On the Pi, Whistle's MT RTF is much larger than its ASR RTF, so fast recognition alone
does not make the cascade fast. Eight-second chunks improve WER, chrF2 and COMET on
both devices but increase measured delay. The larger laptop ASR RTF at 8 seconds
and the Pi MT cost were observed without a resolved causal explanation.

## 8. Inspect the exploratory QC separately

Open `quality-control/TinyML_translation_QC.ipynb` next. It traces the main experiment's saved ASR-to-MT judgments and preserves unsuccessful reviews. This distinction matters: faithful translation of an ASR error can still be wrong relative to the original speech. The supplement received automated metrics only.

Six independent talks and two passes support descriptive comparisons; the walkthrough does not add new experimental observations.


## Appendix: implementation and the evaluation repair

The examples and results above can be read without these source listings. The cells
below display the preserved experimental implementation, they do not execute it.

### A. Audio scheduling and model calls

`replay` uses one continuous clock to schedule arriving chunks and commit translations.
`Cascade` consumes the lazy transcription iterator inside the timed call so ASR work
is included. Its low-level MT beam default is overridden by the experiment wrapper.

In [12]:
show_source("TinyML-main-experiment-2026-10-18/runner/stream_timing.py", "replay")


def replay(wav, chunk_s, infer, emit, *, clock=time.perf_counter, sleep=time.sleep):
    """Read a chunk only after its last sample arrives; never skip overdue chunks.

    infer accepts PCM bytes and returns a dict containing english/german strings.
    emit is called once per completed chunk, including chunks with no output.
    The consumer receives an entire chunk translation atomically.
    """
    if not math.isfinite(chunk_s) or not 0 < chunk_s <= 30:
        raise ValueError("chunk_s must be finite and in (0, 30]")
    rate = wav.getframerate()
    chunk_frames = max(1, round(chunk_s * rate))
    total_frames = wav.getnframes()
    origin = clock()
    pos = 0
    index = 0
    while pos < total_frames:
        end = min(pos + chunk_frames, total_frames)
        ready_s = end / rate
        # A single clock is essential: sleep(chunk_s) after each inference is wrong.
        while (wait_s := ready_s - (clock() - origin)) > 0:
            sleep(wait_s)
        started = clock()
        raw = wav.readframes(end - pos)
        expected = (end - pos) * wav.getsampwidth() * wav.getnchannels()
        if len(raw) != expected:
            raise ValueError("WAV ended before its declared frame count")
        result = infer(raw)
        result["words"] = result["german"].split()
        committed = clock()
        record = dict(result)
        record.update(
            chunk_index=index,
            source_start_s=pos / rate,
            source_end_s=ready_s,
            chunk_duration_s=(end - pos) / rate,
            processing_start_s=started - origin,
            emitted_s=committed - origin,
            process_s=committed - started,
            queue_wait_s=max(0.0, started - origin - ready_s),
            lag_after_chunk_end_s=committed - origin - ready_s,
        )
        emit(record)
        pos = end
        index += 1
    return clock() - origin

In [13]:
show_source("TinyML-main-experiment-2026-10-18/runner/run_pilot.py", "Cascade")


class Cascade:
    def __init__(self, asr_path, mt_path, threads, mt_beam=1):
        import ctranslate2
        from faster_whisper import WhisperModel
        from transformers import MarianTokenizer
        self.asr = WhisperModel(str(asr_path), device="cpu", compute_type="int8",
                                cpu_threads=threads, num_workers=1, local_files_only=True)
        self.tokenizer = MarianTokenizer.from_pretrained(str(mt_path / "tokenizer"),
                                                         local_files_only=True)
        self.mt = ctranslate2.Translator(str(mt_path), device="cpu", compute_type="int8",
                                         inter_threads=1, intra_threads=threads)
        self.mt_beam = mt_beam

    def transcribe(self, audio):
        segments, _ = self.asr.transcribe(
            audio, language="en", task="transcribe", beam_size=1, best_of=1,
            temperature=0.0, condition_on_previous_text=False,
            vad_filter=False, word_timestamps=False, without_timestamps=True,
            no_speech_threshold=0.6,
        )
        # faster-whisper is lazy: iteration MUST remain inside the timed call.
        return " ".join(s.text.strip() for s in segments if s.text.strip())

    def translate(self, english):
        if not english.strip():
            return "", 0, False
        ids = self.tokenizer.encode(english, add_special_tokens=True, truncation=False)
        if len(ids) > 512:
            raise ValueError("ASR text exceeds 512 MT input tokens; refusing silent truncation")
        tokens = self.tokenizer.convert_ids_to_tokens(ids)
        result = self.mt.translate_batch([tokens], beam_size=self.mt_beam, max_batch_size=1,
                                         max_input_length=0, max_decoding_length=256)[0]
        target = result.hypotheses[0]
        german = self.tokenizer.decode(self.tokenizer.convert_tokens_to_ids(target),
                                       skip_special_tokens=True)
        german = " ".join(german.split())
        return german, len(target), len(target) >= 256

    def __call__(self, pcm):
        import numpy as np
        start = time.perf_counter()
        audio = np.frombuffer(pcm, dtype="<i2").astype(np.float32) / 32768.0
        english = self.transcribe(audio)
        asr_end = time.perf_counter()
        german, token_count, hit_limit = self.translate(english)
        mt_end = time.perf_counter()
        return dict(english=english, german=german, asr_s=asr_end - start,
                    mt_s=mt_end - asr_end, mt_tokens=token_count, mt_limit_reached=hit_limit)

### B. Why empty timestamp lists were needed

**Some reference segments received no translated words, so their timestamp lists
should be empty.** Resegmentation instead left those lists missing or null, which
the original evaluator rejected. A zero-word prediction needs zero timestamps for
each timing convention: `emission_cu = []` and `emission_ca = []`.

The separate adapter below supplies empty lists only for zero-word predictions.
Nonempty text without valid timestamps remains an error. Empty predictions stay in
quality scoring; unavailable latency is not replaced with zero. The saved repair
record uses the implementation term “arrays” for these lists of output-word timestamps.

In [14]:
display(JSON(scores["evaluation_fix"], expanded=True))
show_source("evaluation/evaluate_main_fixed.py", "normalize_empty_timestamps")

<IPython.core.display.JSON object>

def normalize_empty_timestamps(rows):
    normalized, changes = [], []
    for i, original in enumerate(rows):
        row = dict(original)
        prediction = row.get("prediction")
        if not isinstance(prediction, str):
            raise ValueError(f"Resegmented row {i}: missing or invalid prediction")
        words = len(prediction.split())
        added = []
        for key in ("emission_cu", "emission_ca"):
            values = row.get(key)
            if values is None:
                if words:
                    raise ValueError(f"Resegmented row {i}: missing {key} for {words} output words; refusing to invent timestamps")
                row[key] = []
                added.append(key)
            elif not isinstance(values, list) or len(values) != words:
                raise ValueError(f"Resegmented row {i}: {key} count does not match output words")
            elif any(isinstance(v, bool) or not isinstance(v, (int, float)) or not math.isfinite(v) for v in values):
                raise ValueError(f"Resegmented row {i}: invalid numeric value in {key}")
        if added:
            changes.append({"row_0_based": i, "fields_set_to_empty_arrays": added})
        normalized.append(row)
    return normalized, changes